# Les dates des tirelires, et melanger plusieurs tableaux (explique simplement)

On reprend le tableau des tirelires (les comptes bancaires de Beobank).

Ce notebook a deux parties :

1. **Les dates** : comment lire, comparer, trier et calculer avec des dates dans un
   tableau Pandas.
2. **Melanger plusieurs tableaux** : la banque n'a pas qu'un seul tableau. Il y a
   aussi un tableau des **copains** (les personnes qui possedent les tirelires) et un
   tableau de leurs **adresses**. On va apprendre a coller ces tableaux ensemble.

## 0. On ouvre et on nettoie le tableau des tirelires

Comme d'habitude : on ouvre le fichier, on donne des noms simples aux colonnes.

In [1]:
import pandas as pd   # la boite a outils pour les tableaux
import numpy as np    # la boite a outils pour les calculs

tirelires = pd.read_csv("../data/CTR.csv", sep=";", encoding="cp1252")

tirelires = tirelires.rename(columns={
    "IDT_AC": "numero_tirelire",
    "DAT_OUV_CTR": "date_ouverture",     # le jour ou la tirelire a ete ouverte
    "DAT_CLO_CTR": "date_fermeture",     # le jour ou la tirelire a ete fermee (si elle l'est)
    "DAT_ECV_CTR": "date_evenement",     # le jour du dernier evenement sur la tirelire
    "DAT_MAJ_SLD": "date_maj_solde",     # le jour ou on a compte l'argent pour la derniere fois
    "COD_DEV": "monnaie",
    "SLD_CTR": "argent_dedans",
    "COD_ECV_CTR": "code_statut",
})

# "." veut dire "on ne sait pas" dans ce fichier -> on le remplace par une vraie case vide
tirelires = tirelires.replace(".", pd.NA)
tirelires["argent_dedans"] = pd.to_numeric(tirelires["argent_dedans"], errors="coerce")

tirelires.head(3)

,numero_tirelire,REF_CTR_INN,date_ouverture,code_statut,date_evenement,date_fermeture,monnaie,argent_dedans,date_maj_solde,SLD_DSP,MNT_INI
0,65500004701,29862201102,2024-05-29,6,2025-12-10,2025-12-10,EUR,NaN,NaN,NaN,NaN
1,65500006391,29912218433,2024-08-07,6,2025-02-05,2025-02-05,EUR,NaN,NaN,NaN,NaN
2,65500007774,29922113324,2022-01-12,4,2022-01-12,NaN,EUR,NaN,NaN,NaN,NaN


## 1. Transformer du texte en vraie date

Dans le fichier, une date comme `"2024-05-29"` n'est **pas** une vraie date pour Pandas : c'est juste du texte (des lettres). Pandas ne peut pas faire de calcul avec du texte.

`pd.to_datetime()` transforme le texte en une vraie date, que Pandas sait comparer, trier et compter.

In [2]:
print(tirelires["date_ouverture"].dtype)   # avant : "str" -> c'est juste du texte

# pd.to_datetime(colonne, errors="coerce") : transforme le texte en vraie date
# errors="coerce" : si ce n'est pas une date valable (ou si c'est vide), on met une case vide (NaT)
tirelires["date_ouverture"] = pd.to_datetime(tirelires["date_ouverture"], errors="coerce")
tirelires["date_fermeture"] = pd.to_datetime(tirelires["date_fermeture"], errors="coerce")
tirelires["date_evenement"] = pd.to_datetime(tirelires["date_evenement"], errors="coerce")
tirelires["date_maj_solde"] = pd.to_datetime(tirelires["date_maj_solde"], errors="coerce")

print(tirelires["date_ouverture"].dtype)   # maintenant : "datetime64" -> une vraie date !
tirelires[["numero_tirelire", "date_ouverture", "date_fermeture"]].head()

str
datetime64[us]


,numero_tirelire,date_ouverture,date_fermeture
0,65500004701,2024-05-29,2025-12-10
1,65500006391,2024-08-07,2025-02-05
2,65500007774,2022-01-12,NaT
3,65500008787,2025-01-07,NaT
4,65500014230,2025-01-18,2025-02-15


**NaT** = *Not a Time* = la version "case vide" pour les dates, comme `NaN` pour les nombres. Les tirelires encore ouvertes ont un `NaT` dans `date_fermeture` (puisqu'elles n'ont pas encore ete fermees).

## 2. Regarder les morceaux d'une date : `.dt`

Une fois que c'est une vraie date, on peut demander a Pandas juste l'annee, juste le mois, juste le jour... avec `.dt` (comme `.str` pour le texte).

In [3]:
tirelires["annee_ouverture"] = tirelires["date_ouverture"].dt.year      # l'annee : 2024
tirelires["mois_ouverture"] = tirelires["date_ouverture"].dt.month     # le mois : 5 (pour mai)
tirelires["jour_ouverture"] = tirelires["date_ouverture"].dt.day       # le jour du mois : 29
tirelires["jour_semaine"] = tirelires["date_ouverture"].dt.day_name()  # le nom du jour : "Wednesday"

tirelires[["date_ouverture", "annee_ouverture", "mois_ouverture", "jour_ouverture", "jour_semaine"]].head()

,date_ouverture,annee_ouverture,mois_ouverture,jour_ouverture,jour_semaine
0,2024-05-29,2024,5,29,Wednesday
1,2024-08-07,2024,8,7,Wednesday
2,2022-01-12,2022,1,12,Wednesday
3,2025-01-07,2025,1,7,Tuesday
4,2025-01-18,2025,1,18,Saturday


## 3. Calculer l'age d'une tirelire

On peut soustraire deux dates : le resultat est une **duree** (`Timedelta`), pas une date. `pd.Timestamp.now()` donne la date et l'heure d'aujourd'hui.

In [4]:
aujourdhui = pd.Timestamp.now()

# date - date = une DUREE (pas une date)
tirelires["age_en_jours"] = (aujourdhui - tirelires["date_ouverture"]).dt.days
tirelires["age_en_annees"] = (tirelires["age_en_jours"] / 365).round(1)

tirelires[["date_ouverture", "age_en_jours", "age_en_annees"]].head()

,date_ouverture,age_en_jours,age_en_annees
0,2024-05-29,842,2.3
1,2024-08-07,772,2.1
2,2022-01-12,1710,4.7
3,2025-01-07,619,1.7
4,2025-01-18,608,1.7


## 4. Trier par date

`.sort_values()` marche aussi avec des dates : les plus vieilles dates d'abord, ou les plus recentes d'abord.

In [5]:
# la tirelire la PLUS VIEILLE (ouverte en premier)
tirelires.sort_values("date_ouverture").head(3)[["numero_tirelire", "date_ouverture"]]

,numero_tirelire,date_ouverture
130,65500344419,2009-08-28
118,65500295573,2009-08-28
48,65500115203,2009-09-02


In [6]:
# la tirelire la PLUS RECENTE (ouverte en dernier) : ascending=False -> du plus grand au plus petit
tirelires.sort_values("date_ouverture", ascending=False).head(3)[["numero_tirelire", "date_ouverture"]]

,numero_tirelire,date_ouverture
192,65500491440,2026-06-27
179,65500491167,2026-03-01
197,65500491523,2026-01-09


## 5. Filtrer avec des dates

On peut comparer une colonne de dates avec `<`, `>`, `==`, exactement comme avec des nombres.

In [7]:
# les tirelires ouvertes en 2025
tirelires_2025 = tirelires[tirelires["annee_ouverture"] == 2025]
print(tirelires_2025.shape[0], "tirelires ouvertes en 2025")

# les tirelires ouvertes APRES le 1er janvier 2025
recentes = tirelires[tirelires["date_ouverture"] > "2025-01-01"]
print(recentes.shape[0], "tirelires ouvertes apres le 1er janvier 2025")

# les tirelires ouvertes ENTRE deux dates (.between())
entre_deux = tirelires[tirelires["date_ouverture"].between("2024-01-01", "2024-12-31")]
print(entre_deux.shape[0], "tirelires ouvertes en 2024")

41 tirelires ouvertes en 2025
62 tirelires ouvertes apres le 1er janvier 2025
9 tirelires ouvertes en 2024


## 6. Tirelires ouvertes ou fermees ?

`date_fermeture` est vide (`NaT`) pour une tirelire encore ouverte. `.isna()` le detecte.

In [8]:
encore_ouvertes = tirelires["date_fermeture"].isna()
print(encore_ouvertes.sum(), "tirelires encore ouvertes")
print((~encore_ouvertes).sum(), "tirelires fermees")

# pour les tirelires FERMEES : combien de temps elles sont restees ouvertes ?
fermees = tirelires[~encore_ouvertes].copy()
fermees["duree_ouverte_jours"] = (fermees["date_fermeture"] - fermees["date_ouverture"]).dt.days
fermees[["numero_tirelire", "date_ouverture", "date_fermeture", "duree_ouverte_jours"]].sort_values(
    "duree_ouverte_jours", ascending=False
).head(5)

105 tirelires encore ouvertes
95 tirelires fermees


,numero_tirelire,date_ouverture,date_fermeture,duree_ouverte_jours
49,65500115646,2010-09-29,2026-04-15,5677
34,65500095095,2012-06-21,2025-03-03,4638
26,65500084655,2014-10-15,2025-12-31,4095
27,65500084656,2014-10-15,2025-12-31,4095
92,65500196292,2012-06-21,2023-07-03,4029


## 7. Compter par annee ou par mois

`.value_counts()` compte combien de fois chaque valeur apparait. Tres pratique sur une annee ou un mois.

In [9]:
# combien de tirelires ouvertes chaque annee, triees par annee
tirelires["annee_ouverture"].value_counts().sort_index()

annee_ouverture
2009     7
2010     2
2011     1
2012     5
2013     2
2014    11
2015     6
2016     3
2017     4
2018    17
2020    16
2021    19
2022    21
2023    15
2024     9
2025    41
2026    21
Name: count, dtype: int64

In [10]:
# .groupby() : on fait des paquets par annee, puis on calcule une moyenne par paquet
tirelires.groupby("annee_ouverture")["argent_dedans"].mean().round(2)

annee_ouverture
2009      929.63
2010        0.00
2011    79156.24
2012         NaN
2013        0.00
2014     -222.42
2015        0.00
2016      134.85
2017        0.00
2018   -26620.89
2020    -1065.49
2021   -16847.15
2022   -42005.06
2023   -65288.47
2024   -34158.02
2025    -5442.31
2026        0.00
Name: argent_dedans, dtype: float64

## 8. Ajouter du temps a une date

`pd.Timedelta(...)` est une duree qu'on peut ajouter ou enlever a une date, comme compter des jours sur les doigts.

In [11]:
# la date du prochain anniversaire de la tirelire (1 an apres l'ouverture)
tirelires["prochain_anniversaire"] = tirelires["date_ouverture"] + pd.DateOffset(years=1)

# une date de relance : 30 jours apres la derniere mise a jour du solde
tirelires["date_relance"] = tirelires["date_maj_solde"] + pd.Timedelta(days=30)

tirelires[["date_ouverture", "prochain_anniversaire", "date_maj_solde", "date_relance"]].head()

,date_ouverture,prochain_anniversaire,date_maj_solde,date_relance
0,2024-05-29,2025-05-29,NaT,NaT
1,2024-08-07,2025-08-07,NaT,NaT
2,2022-01-12,2023-01-12,NaT,NaT
3,2025-01-07,2026-01-07,2025-10-31,2025-11-30
4,2025-01-18,2026-01-18,NaT,NaT


## 9. On a d'autres tableaux !

Jusqu'ici on n'a regarde QU'UN tableau : les tirelires. Mais la banque a d'autres
tableaux, comme des cahiers differents qui parlent des memes personnes :

- `TIE.csv` : les **copains** (les personnes), avec leur date de naissance, leur sexe...
- `TIE_X_CTR.csv` : le cahier qui dit **quel copain possede quelle tirelire**
  (le lien entre les deux tableaux)
- `TIE_ADR.csv` : l'**adresse** de chaque copain (pays, ville...)

On va apprendre a les coller ensemble avec `.merge()`, un peu comme on assemble des
pieces de puzzle grace a un numero commun.

In [12]:
copains = pd.read_csv("../data/TIE.csv", sep=";", encoding="cp1252")
copains = copains.rename(columns={
    "NUM_TIE": "numero_copain",
    "DAT_NAI": "date_naissance",
    "DAT_DCS": "date_deces",
    "COD_SEX": "sexe",
    "COD_LNG_CTR": "langue",
})
copains = copains.replace(".", pd.NA)
copains["date_naissance"] = pd.to_datetime(copains["date_naissance"], errors="coerce")

copains[["numero_copain", "date_naissance", "sexe", "langue"]].head(3)

,numero_copain,date_naissance,sexe,langue
0,2500003178436,NaT,NaN,FR
1,2500003178512,2004-03-28,M,FR
2,2500003178544,1980-01-25,M,FR


In [13]:
liaison = pd.read_csv("../data/TIE_X_CTR.csv", sep=";", encoding="cp1252")
liaison = liaison.rename(columns={
    "IDT_AC": "numero_tirelire",
    "NUM_TIE": "numero_copain",
    "COD_ROL_TTL": "role",
})

# ce tableau ne sert qu'a UNE chose : dire quel numero_copain va avec quel numero_tirelire
liaison[["numero_tirelire", "numero_copain", "role"]].head(3)

,numero_tirelire,numero_copain,role
0,65500477817,2500003178544,C
1,65500477836,2500003178544,NaN
2,65500477838,2500003178544,NaN


## 10. `.merge()` : coller deux tableaux grace a une colonne commune

`tirelires` a une colonne `numero_tirelire`. `liaison` a AUSSI une colonne `numero_tirelire`. `.merge(on="...")` dit a Pandas : "pour chaque ligne, va chercher la ligne de l'autre tableau qui a le meme numero, et colle les deux lignes ensemble".

In [14]:
tirelires_et_copains = tirelires.merge(liaison, on="numero_tirelire", how="left")

print("avant le merge :", tirelires.shape)
print("apres le merge :", tirelires_et_copains.shape)

tirelires_et_copains[["numero_tirelire", "numero_copain", "role"]].head()

avant le merge : (200, 19)
apres le merge : (200, 24)


,numero_tirelire,numero_copain,role
0,65500004701,2900000000853,NaN
1,65500006391,2900000000853,C
2,65500007774,2900000004654,C
3,65500008787,2900000004514,NaN
4,65500014230,2900000000853,C


`how="left"` veut dire : **on garde toutes les lignes de tirelires**, meme si jamais une tirelire n'avait pas de copain associe (elle aurait alors une case vide). `how="inner"` ne garderait que les lignes qui existent dans LES DEUX tableaux.

## 11. On merge encore, pour avoir les infos du copain

Meme principe, mais cette fois avec la colonne commune `numero_copain`.

In [15]:
tout = tirelires_et_copains.merge(copains, on="numero_copain", how="left")

# maintenant qu'on a la date de naissance du copain ET la date d'ouverture de la tirelire,
# on peut calculer l'age du copain quand il a ouvert sa tirelire !
tout["age_a_ouverture"] = (
    (tout["date_ouverture"] - tout["date_naissance"]).dt.days / 365
).round(1)

tout[["numero_tirelire", "date_ouverture", "date_naissance", "age_a_ouverture"]].head()

,numero_tirelire,date_ouverture,date_naissance,age_a_ouverture
0,65500004701,2024-05-29,1990-10-28,33.6
1,65500006391,2024-08-07,1990-10-28,33.8
2,65500007774,2022-01-12,1984-10-25,37.2
3,65500008787,2025-01-07,1986-12-03,38.1
4,65500014230,2025-01-18,1990-10-28,34.2


## 12. Un troisieme merge : les adresses

On ajoute le pays et la ville de naissance de chaque copain.

In [16]:
adresses = pd.read_csv("../data/TIE_ADR.csv", sep=";", encoding="cp1252")
adresses = adresses.rename(columns={
    "NUM_TIE": "numero_copain",
    "COD_PAY_ISO": "pays",
    "NOM_CMU_NAI": "ville_naissance",
})

tout = tout.merge(adresses[["numero_copain", "pays", "ville_naissance"]], on="numero_copain", how="left")

tout[["numero_tirelire", "numero_copain", "pays", "ville_naissance", "age_a_ouverture"]].head()

,numero_tirelire,numero_copain,pays,ville_naissance,age_a_ouverture
0,65500004701,2900000000853,BE,CHARLEROI(D 1),33.6
1,65500006391,2900000000853,BE,CHARLEROI(D 1),33.8
2,65500007774,2900000004654,KH,MOUSCRON,37.2
3,65500008787,2900000004514,BE,GENT,38.1
4,65500014230,2900000000853,BE,CHARLEROI(D 1),34.2


## 13. Maintenant qu'on a TOUT dans un seul tableau...

On peut poser des questions qui melangent des dates ET des infos d'un autre tableau, comme :
"Par pays, combien de tirelires, et quel age avaient les copains en moyenne quand ils les ont ouvertes ?"

In [17]:
tout.groupby("pays").agg(
    nombre_de_tirelires=("numero_tirelire", "count"),
    age_moyen_a_ouverture=("age_a_ouverture", "mean"),
).round(1).sort_values("nombre_de_tirelires", ascending=False)

,nombre_de_tirelires,age_moyen_a_ouverture
pays,,
BE,154,44.4
KH,46,34.5


## 14. `pd.concat()` : empiler deux tableaux l'un sous l'autre

`.merge()` colle des tableaux **cote a cote** (en ajoutant des colonnes). `pd.concat()` fait l'inverse : il empile des tableaux **les uns sous les autres** (en ajoutant des lignes), comme deux piles de cartes qu'on met bout a bout. Il faut que les colonnes se ressemblent.

In [ ]:
tirelires_2024 = tout[tout["annee_ouverture"] == 2024]
tirelires_2025 = tout[tout["annee_ouverture"] == 2025]

print("2024 :", tirelires_2024.shape[0], "lignes")
print("2025 :", tirelires_2025.shape[0], "lignes")

# pd.concat([tableau1, tableau2]) : on empile les deux paquets l'un sous l'autre
empilees = pd.concat([tirelires_2024, tirelires_2025])
print("empilees :", empilees.shape[0], "lignes  (", tirelires_2024.shape[0], "+", tirelires_2025.shape[0], ")")

empilees[["numero_tirelire", "annee_ouverture", "pays"]].sort_values("annee_ouverture").head()

## 15. Fiche recap

| Fonction | A quoi ca sert (en tres simple) |
|---|---|
| `pd.to_datetime()` | Transformer du texte en vraie date |
| `.dt.year` / `.dt.month` / `.dt.day` | Recuperer un morceau d'une date |
| `.dt.day_name()` | Le nom du jour de la semaine |
| `date2 - date1` | Calculer une duree entre deux dates |
| `pd.Timestamp.now()` | La date et l'heure d'aujourd'hui |
| `pd.Timedelta(days=...)` / `pd.DateOffset(years=...)` | Ajouter du temps a une date |
| `.between(date_a, date_b)` | Garder les dates entre deux bornes |
| `.sort_values("colonne_date")` | Trier du plus vieux au plus recent (ou l'inverse) |
| `.value_counts()` | Compter combien de fois chaque valeur apparait |
| `df1.merge(df2, on="cle", how="left")` | Coller deux tableaux cote a cote grace a une colonne commune |
| `pd.concat([df1, df2])` | Empiler deux tableaux l'un sous l'autre |

**Bravo ! Tu sais maintenant lire des dates, calculer avec, et assembler plusieurs
tableaux comme un vrai puzzle.**